# ⚖️ NeuralShield-v6 — ADİL (Fair) Kaynak-Ayrık Benchmark

**Amaç:** Patronus Wolf Defender'ın eğitiminde kullandığı `jackhhao` kaynağını **çıkararak** adil kıyas yapmak.

- **Saldırı (EN):** `Simsonsun/JailbreakPrompts` + `rubend18/ChatGPT-Jailbreak-Prompts` (Patronus'un eğitim listesinde YOK)
- **Masum (EN):** `databricks/dolly-15k` (Patronus'un eğitim listesinde YOK)
- NLLB-200 ile 11 dile çevrilir → **12 dil × (120 + 120) = 2.880**
- Çıktı: `NeuralShield_v6_MasterData/heldout_benchmark_v6_fair.json` + `data/heldout_benchmark_v6_fair.json`

Sonra Hücre 3: v6.2 modellerimiz + Patronus'u **adil** kümede test eder.

In [ ]:
# @title 1. Kurulum + Drive
import os, sys, json, re, random, unicodedata, torch, gc, time
import warnings; warnings.filterwarnings('ignore')
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)
MASTER_DIR = '/content/drive/MyDrive/NeuralShield_v6_MasterData'
os.makedirs(MASTER_DIR, exist_ok=True)
!pip -q install -U transformers datasets accelerate sentencepiece 2>/dev/null
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
SEED=42; random.seed(SEED)

In [ ]:
# @title 2. Adil Benchmark üret (Simsonsun + rubend18 / dolly) + NLLB 12 dil
import json, re, random, unicodedata
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from collections import Counter

LANGS = ['en','tr','de','fr','es','it','pt','ru','ar','ja','ko','zh']
NLLB_MAP = {'tr':'tur_Latn','de':'deu_Latn','fr':'fra_Latn','es':'spa_Latn','it':'ita_Latn',
            'pt':'por_Latn','ru':'rus_Cyrl','ar':'arb_Arab','ja':'jpn_Jpan','ko':'kor_Hang','zh':'zho_Hans'}
ETHICS = re.compile(r'(suicide|intihar|genocide|soyk[ıi]r|organ mafya|ırkç|nefret söylemi|9/11|nuke|bomb|bomba|meth|heroin|ricin|sarin|nerve agent|kimyasal silah|uyuşturucu|fidye)', re.I)
def clean(t, lo=15, hi=1200):
    t=str(t or '').strip()
    return t if (lo<=len(t)<=hi and not ETHICS.search(t)) else None

en_atk, en_ben = [], []
# SALDIRI 1: Simsonsun (Dataset_1+Dataset_2)
ds = load_dataset('Simsonsun/JailbreakPrompts', split='Dataset_1+Dataset_2')
for r in ds:
    t=clean(r.get('Prompt') or r.get('text') or r.get('prompt'))
    if t: en_atk.append((t,'HF/Simsonsun/JailbreakPrompts'))
# SALDIRI 2: rubend18
for r in load_dataset('rubend18/ChatGPT-Jailbreak-Prompts', split='train'):
    t=clean(r.get('Prompt'))
    if t: en_atk.append((t,'HF/rubend18'))
# MASUM: dolly
for r in load_dataset('databricks/databricks-dolly-15k', split='train'):
    t=clean(r.get('instruction'))
    if t: en_ben.append((t,'HF/dolly-15k'))

def dedup_shuffle(pairs):
    seen=set(); out=[]
    for t,s in pairs:
        k=re.sub(r'\s+',' ',unicodedata.normalize('NFKC',t).lower().strip())
        if k in seen: continue
        seen.add(k); out.append((t,s))
    random.Random(SEED).shuffle(out); return out
en_atk=dedup_shuffle(en_atk); en_ben=dedup_shuffle(en_ben)
print('EN havuz: atk=',len(en_atk),'ben=',len(en_ben))
A=en_atk[:120]; B=en_ben[:120]
print('Secilen: atk=',dict(Counter(s for _,s in A)),'| ben=',dict(Counter(s for _,s in B)))

rows=[]
for t,s in A: rows.append({'text':t,'label':1,'language':'en','source':s})
for t,s in B: rows.append({'text':t,'label':0,'language':'en','source':s})

# NLLB 11 dil
mid='facebook/nllb-200-distilled-600M'
tok=AutoTokenizer.from_pretrained(mid, src_lang='eng_Latn')
mod=AutoModelForSeq2SeqLM.from_pretrained(mid, torch_dtype=torch.float16).to('cuda').eval()
def translate(texts, lg, bs=128):
    out=[]; bos=tok.convert_tokens_to_ids(NLLB_MAP[lg])
    for i in range(0,len(texts),bs):
        batch=[x[:600] for x in texts[i:i+bs]]
        inp=tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to('cuda')
        with torch.no_grad():
            g=mod.generate(**inp, forced_bos_token_id=bos, max_new_tokens=256, num_beams=1)
        out.extend(tok.batch_decode(g, skip_special_tokens=True))
    return [x.strip() for x in out]
src=[t for t,_ in A]; srclb=[s for _,s in A]
srcb=[t for t,_ in B]; srcbl=[s for _,s in B]
for lg in ['tr','de','fr','es','it','pt','ru','ar','ja','ko','zh']:
    t0=time.time()
    ta=translate(src,lg); tb=translate(srcb,lg)
    for t,s in zip(ta,srclb):
        if t and len(t)>=10: rows.append({'text':t,'label':1,'language':lg,'source':s+'_nllb_'+lg})
    for t,s in zip(tb,srcbl):
        if t and len(t)>=10: rows.append({'text':t,'label':0,'language':lg,'source':s+'_nllb_'+lg})
    print(f'  [{lg}] ok ({time.time()-t0:.1f}s)')
del mod,tok; gc.collect(); torch.cuda.empty_cache()

json.dump(rows, open('/content/drive/MyDrive/NeuralShield_v6_MasterData/heldout_benchmark_v6_fair.json','w',encoding='utf-8'), ensure_ascii=False, indent=1)
print('KAYDEDILDI Drive:', len(rows), '| dagilim:', dict(Counter(x['language'] for x in rows)))

In [ ]:
# @title 3. ADİL kıyas: v6.2 modellerimiz + Patronus (+ v6.0 yok, checkpointler güncel)
import json, torch, gc, time, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
data=json.load(open('/content/drive/MyDrive/NeuralShield_v6_MasterData/heldout_benchmark_v6_fair.json',encoding='utf-8'))
texts=[x['text'] for x in data]; labels=[int(x['label']) for x in data]

def eval_ours(name, path, bs=64):
    tok=AutoTokenizer.from_pretrained(path); mod=AutoModelForSequenceClassification.from_pretrained(path, torch_dtype=torch.float16).to('cuda').eval()
    probs=[]
    for i in range(0,len(texts),bs):
        inp=tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
        with torch.no_grad(): probs.extend(torch.softmax(mod(**inp).logits.float(),-1)[:,1].cpu().numpy().tolist())
    del mod,tok; gc.collect(); torch.cuda.empty_cache(); return probs
def eval_rival(name, mid, bs=64):
    clf=pipeline('text-classification', model=mid, truncation=True, max_length=2048, device=0, torch_dtype=torch.float16)
    res=clf(texts, batch_size=bs)
    pos=('INJECTION','LABEL_1','ATTACK','UNSAFE','JAILBREAK','MALICIOUS')
    probs=[r['score'] if any(p in str(r['label']).upper() for p in pos) else 1-r['score'] for r in res]
    del clf; gc.collect(); torch.cuda.empty_cache(); return probs

def get_best_ckpt(base_dir):
    if not os.path.exists(base_dir): return None
    ck_dir = f"{base_dir}/checkpoints"
    search_dir = ck_dir if os.path.exists(ck_dir) else base_dir
    subs = [f"{search_dir}/{d}" for d in os.listdir(search_dir) if d.startswith("checkpoint")]
    if subs:
        return max(subs, key=lambda p: os.path.getmtime(p))
    if any(os.path.exists(f"{base_dir}/{f}") for f in ["config.json", "model.safetensors"]):
        return base_dir
    return None

rows=[]
v62_dir = '/content/drive/MyDrive/NeuralShield_v6_mmbert/checkpoints_v6_2_backup' if os.path.exists('/content/drive/MyDrive/NeuralShield_v6_mmbert/checkpoints_v6_2_backup') else '/content/drive/MyDrive/NeuralShield_v6_mmbert_v62'
cand_models = [
    ('mmBERT v6.3 (Yeni)','/content/drive/MyDrive/NeuralShield_v6_mmbert'),
    ('mmBERT v6.2 (Eski)', v62_dir),
    ('XLM-R v6.2','/content/drive/MyDrive/NeuralShield_v6_xlmr'),
    ('BGE-M3 v6.2','/content/drive/MyDrive/NeuralShield_v6_bgem3')
]
for name, bdir in cand_models:
    ckpt = get_best_ckpt(bdir)
    if ckpt and os.path.exists(ckpt):
        print(f'Yükleniyor: {name} -> {ckpt}')
        p=eval_ours(name,ckpt); rows.append((name,p))
    else:
        print(f'⚠️ Bulunamadı: {name} ({bdir})')
for name,mid in [('Patronus Wolf Defender','patronus-studio/wolf-defender-prompt-injection')]:
    try:
        p=eval_rival(name,mid); rows.append((name,p))
    except Exception as e: print(name,'atlandi',str(e)[:80])

out=[]
for name,p in rows:
    p=np.array(p); pr=(p>=0.5).astype(int); pr7=(p>=0.7).astype(int)
    def m(pv):
        a=[i for i,y in enumerate(labels) if y==1]; b=[i for i,y in enumerate(labels) if y==0]
        rec=100*sum(pv[i] for i in a)/max(1,len(a)); fpr=100*sum(pv[i] for i in b)/max(1,len(b))
        return round(rec,1),round(fpr,1),round(100*f1_score(labels,pv,average='macro'),1)
    r5,f5,q5=m(pr); r7,f7,q7=m(pr7)
    out.append({'Model':name,'Rec@0.5':r5,'FP@0.5':f5,'F1@0.5':q5,'Rec@0.7':r7,'FP@0.7':f7,'F1@0.7':q7,'AUC':round(roc_auc_score(labels,p),4)})
display(pd.DataFrame(out))
for name,p in rows:
    p=np.array(p); pr=(p>=0.7).astype(int)
    print('\n',name,'per-lang (rec/fp @0.7):')
    for lg in ['en','tr','de','fr','es','it','pt','ru','ar','ja','ko','zh']:
        a=[i for i,x in enumerate(data) if x['language']==lg and x['label']==1]
        b=[i for i,x in enumerate(data) if x['language']==lg and x['label']==0]
        rec=round(100*sum(pr[i] for i in a)/max(1,len(a)),1); fp=round(100*sum(pr[i] for i in b)/max(1,len(b)),1)
        print(f'  {lg}: {rec}% / {fp}%')
